<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/04-groupby-agg.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 3.4 — groupBy + agg

Includes the shuffle-partitions demo — run with the UI open.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("3.4-groupby-agg")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
    .withColumn("line_total", col("quantity") * col("unit_price"))
)

## The canonical shape

In [ ]:
print(type(orders.groupBy("category")))   # GroupedData — half a sentence

summary = (
    orders.groupBy("category")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.sum("line_total"), 2).alias("revenue"),
        F.round(F.avg("unit_price"), 2).alias("avg_price"),
        F.countDistinct("customer_id").alias("customers"),
    )
)
summary.orderBy(F.col("revenue").desc()).show()

In [ ]:
# Multi-key grouping + global aggregate (no groupBy)
orders.groupBy("country", "category").agg(F.count("*").alias("n")).orderBy("country", "category").show(8)
orders.agg(F.round(F.sum("line_total"), 2).alias("total_revenue")).show()

## count(*) vs count(col) — nulls, and the first() trap

In [ ]:
orders.agg(
    F.count("*").alias("rows"),
    F.count("quantity").alias("non_null_quantity"),   # skips the 2 nulls
    F.count("country").alias("non_null_country"),     # skips the 3 nulls
).show()

# first() = SOME value, not 'earliest' — run twice, trust nothing:
orders.groupBy("customer_id").agg(F.first("product")).filter(col("customer_id") == "c001").show()

## collect_list / collect_set

In [ ]:
orders.groupBy("customer_id").agg(
    F.collect_list("category").alias("all_cats"),   # keeps duplicates + order-of-arrival
    F.collect_set("category").alias("uniq_cats"),   # deduped, unordered
).filter(F.size("all_cats") > 1).show(5, truncate=False)

## The engine does reduceByKey for you — see the partial aggregation

Two `HashAggregate` nodes around the `Exchange` = combine before shuffle,
finish after. Lesson 2.2's lesson, automated.

In [ ]:
orders.groupBy("category").agg(F.sum("line_total")).explain()

## The 200-shuffle-partitions demo — watch the task counts

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")   # disable AQE so the raw knob shows

spark.conf.set("spark.sql.shuffle.partitions", "200")
orders.groupBy("category").count().collect()
print("-> check the last job's 2nd stage in the UI: 200 tasks for 5 rows")

spark.conf.set("spark.sql.shuffle.partitions", "8")
orders.groupBy("category").count().collect()
print("-> and now: 8 tasks")

spark.conf.set("spark.sql.adaptive.enabled", "true")    # restore

## pivot — crosstab, with the value list pinned

In [ ]:
orders.groupBy("category") \
    .pivot("country", ["IN", "US", "DE", "UK"]) \
    .agg(F.round(F.sum("line_total"), 2)) \
    .show()

## Exercises

1. Revenue, order count, and average line_total per `country` — nulls included as their own group? Check what `groupBy("country")` did with the 3 null countries.
2. Per category: min and max `unit_price` AND the product names achieving them. (You'll discover you can't do the names part cleanly with groupBy — write down why; Module 8 solves it.)
3. Which day had the highest revenue? (`groupBy("order_date")`, top-1 via the 3.3 pattern.)
4. Verify the collect_list OOM logic at toy scale: for `groupBy("country")`, compare the size of `collect_list("product")` output vs `count("product")` output using `F.size`. Which grows with data volume?

In [ ]:
# your exercise code here
